# Question 15 - 135. Candy

There are `n` children standing in a line. Each child is assigned a rating value given in the integer array `ratings`.

You are giving candies to these children subjected to the following requirements:

- Each child must have at least one candy.
- Children with a higher rating get more candies than their neighbors.

Return *the **minimum** number of candies you need to have to distribute the candies to the children*.

**Example 1:**

    Input: ratings = [1,0,2]
    Output: 5
    Explanation: You can allocate to the first, second and third child with 2, 1, 2 candies respectively.

**Example 2:**

    Input: ratings = [1,2,2]
    Output: 4
    Explanation: You can allocate to the first, second and third child with 1, 2, 1 candies respectively.
    The third child gets 1 candy because it satisfies the above two conditions.

**Constraints:**

- `n == ratings.length`
- `1 <= n <= 2 * 10^4`
- `0 <= ratings[i] <= 2 * 10^4`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two passes: satisfy left neighbours, then right neighbours, keep the max

📘 **Revise first:** [Pattern D · prefix & suffix (two passes)](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (fix violations until stable) | O(n²) | O(n) |
| 2️⃣ Optimized (left pass + right pass) | O(n) | O(n) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Children stand in a line, each with a rating. Hand out candy so that:
1. **Everyone gets at least 1.**
2. **If a child's rating is higher than a neighbour's, that child gets more candy than that neighbour.**

Use as **few candies as possible** in total.

```
ratings = [1, 0, 2]
candies = [2, 1, 2]   → total 5
```
The middle child (rating 0) gets the minimum, 1. Both outer children outrank the middle, so each needs more than 1, which means 2.

⚠️ **Equal ratings have no rule.** `[1, 2, 2]` → `[1, 2, 1]`: the last child ties with their neighbour, so they can get just 1.

**Everyday picture: each child looks both ways.** Every child has two possible complaints: *"I outrank the kid on my **left** but don't have more!"* and *"I outrank the kid on my **right** but don't have more!"*

The trick is to handle these **one direction at a time**:
- Walk **left → right** and fix only the "left" complaints.
- Walk **right → left** and fix only the "right" complaints, without breaking what you already fixed (take the bigger of the two requirements).

### Step 0 · Clarify before coding

🗣️ *"Every child gets at least one candy, and a child with a strictly higher rating than an immediate neighbour must get strictly more than that neighbour. I want the minimum total."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Equal ratings: any rule? | **No.** | People often wrongly force equal candies. `[1,2,2]` → 4, not 5. |
| Only immediate neighbours? | **Yes**, left and right. | |
| Return the distribution or the total? | **Total.** | |
| Size? | Up to 2·10⁴. | O(n²) ≈ 4·10⁸ is too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** give everyone 1 candy. Then walk the line and **fix any complaint** you see by bumping the child up to "neighbour + 1". Keep sweeping until a full sweep fixes nothing.

🗣️ *"The simplest correct approach is to start everyone at one and repeatedly fix violations until none are left. Each fix raises a child only to the smallest value that satisfies the rule, so the result is minimal."*

**Why is it slow?** A long **decreasing** line like `[5, 4, 3, 2, 1]` needs the candy counts to build up from the right end, and each sweep only pushes the fix one step. That's up to n sweeps × n children = **O(n²)**.

In [1]:
class SolutionBrute:
    def candy(self, ratings: list[int]) -> int:
        n = len(ratings)
        candies = [1] * n
        changed = True
        while changed:                     # sweep until nothing changes
            changed = False
            for i in range(n):
                if i > 0 and ratings[i] > ratings[i - 1] and candies[i] <= candies[i - 1]:
                    candies[i] = candies[i - 1] + 1
                    changed = True
                if i < n - 1 and ratings[i] > ratings[i + 1] and candies[i] <= candies[i + 1]:
                    candies[i] = candies[i + 1] + 1
                    changed = True
        return sum(candies)

### Step 2 · Optimize: the "aha" moment

Brute force is slow because each child has **two** rules pulling in different directions, and fixing one can break the other, so we bounce back and forth.

> 💡 **Aha:** handle each direction in **the one pass where it's easy**.
> - **Left → right:** when you reach child `i`, the left neighbour's count is already final for the "beat my left neighbour" rule. So: rating went up → `candies[i] = candies[i−1] + 1`, otherwise leave it at 1.
> - **Right → left:** same idea for the "beat my right neighbour" rule. But child `i` may already need a big number from the first pass, so take **`max(current, right neighbour + 1)`**.

Taking the **max** satisfies both rules, and it's the **smallest** number that does, because each pass gave the minimum its own rule required.

🗣️ *"Each child has a left-neighbour rule and a right-neighbour rule. A single left-to-right pass handles every left rule: if the rating goes up, give one more than the left neighbour. A right-to-left pass handles every right rule. Where both rules apply, I take the max of the two requirements, which satisfies both with the fewest candies. Two linear passes, so O(n) time, O(n) space for the candy array."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Fix violations until stable | ❌ | O(n²) on long decreasing runs. |
| Sort children by rating, assign low to high | ⚠️ | Correct (each child gets 1 + the max of lower-rated neighbours), but O(n log n) for the sort. |
| **Two passes + max** | ✅ **best** | O(n), easy to explain and prove. One pass per direction. |
| One pass counting up-slopes and down-slopes | ✅ advanced | O(1) extra memory, but the bookkeeping at peaks is tricky. Mention it, don't lead with it. |

**Why overwriting instead of taking `max` would be wrong:** a peak child might need 4 from the left (a staircase 1, 2, 3, 4) but only 2 from the right. Overwriting with 2 would break the left rule.

In [2]:
class Solution:
    def candy(self, ratings: list[int]) -> int:
        n = len(ratings)
        candies = [1] * n                  # everyone starts with 1
        # pass 1: "beat my LEFT neighbour if I outrank them"
        for i in range(1, n):
            if ratings[i] > ratings[i - 1]:
                candies[i] = candies[i - 1] + 1
        # pass 2: "beat my RIGHT neighbour", without breaking pass 1
        for i in range(n - 2, -1, -1):
            if ratings[i] > ratings[i + 1]:
                candies[i] = max(candies[i], candies[i + 1] + 1)
        return sum(candies)

### Step 3 · Toy example walkthrough (tell it like a story)

`ratings = [1, 3, 4, 5, 2]` (a staircase up to a peak, then a drop)

| | child 0 | child 1 | child 2 | child 3 | child 4 |
|---|---|---|---|---|---|
| rating | 1 | 3 | 4 | 5 | 2 |
| start | 1 | 1 | 1 | 1 | 1 |
| after left → right | 1 | 2 | 3 | **4** | 1 |
| right → left needs | | | | 5 > 2 → 1+1 = 2 | |
| after max | 1 | 2 | 3 | max(4, 2) = **4** | 1 |

Total = 1 + 2 + 3 + 4 + 1 = **11** ✅

**Narrated:** "Going left to right, the ratings climb 1, 3, 4, 5, so the candies climb 1, 2, 3, 4. The last child, rating 2, drops, so they stay at 1. Going back right to left, only child 3 outranks their right neighbour, and they need at least 2, but they already have 4, so I keep 4. Total 11."

**Equal ratings:** `[1, 2, 2]` → left pass `[1, 2, 1]` (the tie doesn't count as "higher", so the last child resets to 1). The right pass changes nothing. Total **4** ✅

In [3]:
cases = [
    ([1, 0, 2], 5),
    ([1, 2, 2], 4),
    ([1], 1),
    ([1, 3, 4, 5, 2], 11),
    ([5, 4, 3, 2, 1], 15),       # 5+4+3+2+1
    ([1, 2, 3, 4, 5], 15),
    ([2, 2, 2], 3),
    ([1, 3, 2, 2, 1], 7),
]
for ratings, expected in cases:
    assert SolutionBrute().candy(ratings) == expected
    assert Solution().candy(ratings) == expected
    print(ratings, "->", expected)

import random
for _ in range(500):
    r = [random.randint(0, 4) for _ in range(random.randint(1, 10))]
    assert Solution().candy(r) == SolutionBrute().candy(r), r
print("All tests passed ✅")

[1, 0, 2] -> 5
[1, 2, 2] -> 4
[1] -> 1
[1, 3, 4, 5, 2] -> 11
[5, 4, 3, 2, 1] -> 15
[1, 2, 3, 4, 5] -> 15
[2, 2, 2] -> 3
[1, 3, 2, 2, 1] -> 7
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Fix until stable | O(n²) | O(n) | Up to n sweeps over n children. |
| Sort by rating | O(n log n) | O(n) | The sort dominates. |
| **Two passes** | **O(n)** | **O(n)** | Two walks over the line plus a sum. |
| Slope counting | O(n) | O(1) | Counts climbs and drops; tricky at peaks. |

**Why O(n) time?** Three straight passes (left, right, sum), each touching every child once.

**Why O(n) memory?** The `candies` array has one slot per child.

**Edge cases to mention:** equal neighbours (no constraint), strictly decreasing (n + (n−1) + … + 1), strictly increasing (same total), a single child (1).

**Likely follow-up:** *"Can you do it in O(1) extra space?"* → Walk once, tracking the length of the current **up** run, the current **down** run, and the height of the last **peak**. A down run of length d costs 1 + 2 + … + d, and when the down run outgrows the peak, the peak needs one more. Explain the idea; code it only if asked.

---

#### 🗣️ Full interview script (about 60 seconds)

*"Every child needs at least one candy, and a child rated higher than a neighbour needs more than that neighbour. Equal ratings impose nothing.*

*The naive approach starts everyone at one and keeps fixing violations until there are none, but long decreasing runs make that O(n²).*

*The key is that each child has two independent constraints, one from the left and one from the right. A left-to-right pass handles all the left constraints: if the rating rises, give one more than the left neighbour. A right-to-left pass handles all the right constraints, and where a child already has a larger count from the first pass, I keep the max. The max satisfies both constraints and is the smallest value that does.*

*Two linear passes: O(n) time and O(n) space. There's also an O(1)-space slope-counting version if needed."*